# Pandas from First Principles
## Bonus Notebook: End-to-End Real Project

---

### The Scenario

You have just joined **TechMart** — a mid-size electronics retailer operating across 4 regions in India.

The data team gives you three raw datasets:
- `orders` — every customer order placed in 2024
- `products` — product catalog with pricing and categories
- `customers` — customer information

Your manager asks you 8 business questions. Your job: clean the data, combine it, and answer them.

---

### Business Questions to Answer

1. Which **region** generated the most total revenue in 2024?
2. Which **product category** had the highest number of orders?
3. Who are the **top 5 customers** by total spending?
4. Which **month** had the highest revenue — and which had the lowest?
5. What is the **average order value** per region?
6. How many orders were **returned** — and what % of total revenue do returns represent?
7. Which **day of the week** sees the most orders?
8. Create a **quarterly revenue report** (pivot table) showing revenue by region and quarter.

---

### Concepts Used (from all 12 notebooks)

| Step | Concepts |
|---|---|
| Data creation | Dict of lists, `pd.DataFrame` (NB2) |
| Inspection | `.info()`, `.head()`, `.describe()` (NB2) |
| Cleaning | `dropna`, `fillna`, `replace`, `drop_duplicates`, `pd.to_numeric`, `pd.to_datetime` (NB3) |
| Strings & Dates | `.str.strip()`, `.str.title()`, `.dt.month`, `.dt.day_name()` (NB4) |
| Sorting | `sort_values`, `nlargest` (NB5) |
| Functions | `np.where`, `pd.cut` (NB6) |
| GroupBy | `groupby`, `agg`, `transform` (NB7) |
| Merging | `pd.merge` (NB8) |
| Reshaping | `pivot_table` (NB10) |

---

**Let's begin.**

---

## Step 1: Create the Raw Datasets

In a real project, you would do `pd.read_csv('orders.csv')`. Here, we build the datasets directly so you can run this notebook without any files.

We intentionally add mess — missing values, wrong types, duplicates, inconsistent casing — just like real data.

In [ ]:
import pandas as pd
import numpy as np

# ── ORDERS TABLE ──────────────────────────────────────────────────────────────
# Intentional problems: missing values, wrong types, duplicate row, inconsistent status casing
orders_raw = pd.DataFrame({
    "order_id":    [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,
                    1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,
                    1021,1022,1023,1024,1025,1003],  # 1003 is a duplicate
    "customer_id": ["C001","C002","C001","C003","C004","C002","C005","C003","C006","C001",
                    "C004","C007","C002","C008","C005","C001","C009","C003","C007","C010",
                    "C006","C002","C008","C001","C005","C001"],
    "product_id":  ["P01","P02","P03","P01","P04","P03","P02","P05","P01","P04",
                    "P03","P01","P02","P06","P04","P05","P03","P01","P06","P02",
                    "P04","P05","P01","P03","P06","P03"],
    "quantity":    ["2","1","1","3","1","2","1","2","1","1",
                    "2","1","3","1","2","1","1","2","1","1",
                    "3","1","2","1","1","1"],   # stored as string — needs conversion
    "order_date":  ["2024-01-10","2024-01-22","2024-02-05","2024-02-18","2024-03-03",
                    "2024-03-15","2024-04-08","2024-04-20","2024-05-11","2024-05-28",
                    "2024-06-04","2024-06-19","2024-07-07","2024-07-23","2024-08-02",
                    "2024-08-15","2024-09-09","2024-09-21","2024-10-06","2024-10-18",
                    "2024-11-03","2024-11-25","2024-12-05","2024-12-18","2024-12-28","2024-02-05"],
    "status":      ["Delivered","DELIVERED","delivered","Returned","Delivered",
                    "Delivered","RETURNED","Delivered","Delivered","returned",
                    "Delivered","Delivered","Delivered","Delivered","DELIVERED",
                    "Delivered","Delivered","Delivered","Returned","Delivered",
                    "Delivered","Delivered","Returned","Delivered","Delivered","delivered"],
    "discount":    [0,500,0,200,0,0,300,0,0,0,
                    150,0,0,0,200,0,0,100,0,0,
                    0,0,0,50,0,0],
    "region":      ["North","South","East","north","West","SOUTH","East","West","north","South",
                    "North","East","West","South","north","East","West","South","North","East",
                    "West","North","south","East",None,"East"],  # has None and inconsistent casing
})

print("Orders shape:", orders_raw.shape)
print(orders_raw.head(5))

In [ ]:
# ── PRODUCTS TABLE ────────────────────────────────────────────────────────────
products_raw = pd.DataFrame({
    "product_id":  ["P01","P02","P03","P04","P05","P06"],
    "product_name":["  Laptop Pro","Wireless Mouse "," Mechanical Keyboard","4K Monitor","USB-C Hub ","Webcam HD"],
    "category":    ["Laptops","Accessories","Accessories","Monitors","Accessories","Peripherals"],
    "unit_price":  ["75000","1200","2500","22000","3500","4800"],  # stored as string
    "brand":       ["Dell","Logitech","Corsair","LG","Anker","Logitech"],
})

print("\nProducts shape:", products_raw.shape)
print(products_raw)

In [ ]:
# ── CUSTOMERS TABLE ───────────────────────────────────────────────────────────
customers_raw = pd.DataFrame({
    "customer_id":   ["C001","C002","C003","C004","C005","C006","C007","C008","C009","C010"],
    "customer_name": ["Arjun Sharma","Priya Mehta","Rahul Singh","  neha gupta","VIKRAM NAIR",
                      "Sneha Patel","Aditya Kumar","Meera Joshi","Rohan Das","Kavya Reddy"],
    "city":          ["Delhi","Mumbai","Pune","Bangalore","Chennai","Delhi","Hyderabad","Mumbai","Kolkata","Bangalore"],
    "age":           [28, 34, 41, 26, 38, 31, 45, 29, 36, 23],
    "member_since":  ["2021-03-15","2020-07-22","2019-11-05","2022-01-10","2021-08-30",
                      "2023-02-14","2020-04-01","2022-09-18","2021-06-25","2023-11-07"],
})

print("\nCustomers shape:", customers_raw.shape)
print(customers_raw)

---

## Step 2: Inspect the Data

Before cleaning anything, understand what you have.

In [ ]:
# Quick inspection of orders
print("=== ORDERS INFO ===")
orders_raw.info()
print("\nMissing values:")
print(orders_raw.isna().sum())

In [ ]:
# Problems we can see:
print("Duplicate orders:", orders_raw.duplicated(subset=['order_id']).sum())
print("Unique status values:", orders_raw['status'].unique())
print("Unique region values:", orders_raw['region'].unique())
print("quantity dtype:", orders_raw['quantity'].dtype,  "← should be int")
print("unit_price dtype:", products_raw['unit_price'].dtype, "← should be int")

**Problems found:**

| Column | Problem |
|---|---|
| `order_id` | 1 duplicate row (order 1003 appears twice) |
| `quantity` | Stored as string, should be int |
| `status` | Inconsistent casing: `'Delivered'`, `'DELIVERED'`, `'delivered'` |
| `region` | Inconsistent casing: `'North'`, `'north'`, `'SOUTH'` + 1 missing value |
| `order_date` | Stored as string, should be datetime |
| `product_name` | Has extra whitespace |
| `customer_name` | Some are uppercase, some have leading spaces |
| `unit_price` | Stored as string, should be int |

Now let's fix everything.

---

## Step 3: Clean the Data

In [ ]:
# ── Clean ORDERS ──────────────────────────────────────────────────────────────

orders = orders_raw.copy()

# 1. Remove duplicate orders (keep first)
orders = orders.drop_duplicates(subset=['order_id'], keep='first')
print(f"After removing duplicates: {len(orders)} rows (was {len(orders_raw)})")

# 2. Convert quantity to integer
orders['quantity'] = pd.to_numeric(orders['quantity'], errors='coerce').fillna(1).astype(int)

# 3. Standardize status — Title Case
orders['status'] = orders['status'].str.title()
print("Unique statuses after fix:", orders['status'].unique())

# 4. Standardize region — Title Case, fill missing with 'Unknown'
orders['region'] = orders['region'].str.title()
orders['region'] = orders['region'].fillna('Unknown')
print("Unique regions after fix:", sorted(orders['region'].unique()))

# 5. Convert order_date to datetime
orders['order_date'] = pd.to_datetime(orders['order_date'])

print("\nOrders dtypes after cleaning:")
print(orders.dtypes)

In [ ]:
# ── Clean PRODUCTS ────────────────────────────────────────────────────────────

products = products_raw.copy()

# 1. Strip whitespace from product_name
products['product_name'] = products['product_name'].str.strip()

# 2. Convert unit_price to integer
products['unit_price'] = pd.to_numeric(products['unit_price'], errors='coerce').astype(int)

print("Products after cleaning:")
print(products)

In [ ]:
# ── Clean CUSTOMERS ───────────────────────────────────────────────────────────

customers = customers_raw.copy()

# 1. Fix customer names — strip whitespace and Title Case
customers['customer_name'] = customers['customer_name'].str.strip().str.title()

# 2. Convert member_since to datetime
customers['member_since'] = pd.to_datetime(customers['member_since'])

print("Customers after cleaning:")
print(customers)

---

## Step 4: Combine the Tables

Right now we have 3 separate tables. We need to merge them into one master table for analysis.

In [ ]:
# Merge orders with products (to get unit_price and category)
orders_products = pd.merge(
    orders,
    products[['product_id', 'product_name', 'category', 'unit_price']],
    on='product_id',
    how='left'
)

print(f"After orders + products merge: {orders_products.shape}")
print(orders_products[['order_id','product_name','category','unit_price']].head())

In [ ]:
# Merge with customers (to get customer name and city)
master = pd.merge(
    orders_products,
    customers[['customer_id', 'customer_name', 'city']],
    on='customer_id',
    how='left'
)

print(f"Master table shape: {master.shape}")
print(master.head(3))

---

## Step 5: Add Derived Columns

Compute new columns we will need for answering the business questions.

In [ ]:
# Gross revenue = unit_price × quantity
master['gross_revenue'] = master['unit_price'] * master['quantity']

# Net revenue = gross_revenue - discount
master['net_revenue'] = master['gross_revenue'] - master['discount']

# Month number and name
master['month']      = master['order_date'].dt.month
master['month_name'] = master['order_date'].dt.strftime('%b')  # Jan, Feb, etc.

# Quarter
master['quarter'] = master['order_date'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})

# Day of week
master['day_of_week'] = master['order_date'].dt.day_name()

# Order size category
master['order_size'] = pd.cut(
    master['net_revenue'],
    bins=[0, 5000, 25000, 100000],
    labels=['Small', 'Medium', 'Large']
)

# Is it a return?
master['is_return'] = master['status'] == 'Returned'

print("Columns in master table:")
print(master.columns.tolist())
print(f"\nShape: {master.shape}")

In [ ]:
# Quick look at the final master table
print(master[['order_id','customer_name','product_name','quantity',
              'net_revenue','month_name','quarter','status','region']].head(8))

---

## Step 6: Answer the Business Questions

All the analysis goes through the cleaned, merged `master` table.

---

### Q1 — Which region generated the most total revenue in 2024?

In [ ]:
# Exclude returns from revenue (returned orders don't count as revenue)
delivered = master[master['status'] == 'Delivered']

revenue_by_region = (
    delivered
    .groupby('region')['net_revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
revenue_by_region.columns = ['region', 'total_revenue']

print("Revenue by Region (Delivered orders only):")
print(revenue_by_region.to_string(index=False))

top_region = revenue_by_region.iloc[0]
print(f"\n✅ Answer: '{top_region['region']}' with ₹{top_region['total_revenue']:,.0f} in revenue")

### Q2 — Which product category had the highest number of orders?

In [ ]:
orders_by_category = (
    master
    .groupby('category')['order_id']
    .count()
    .sort_values(ascending=False)
    .reset_index()
)
orders_by_category.columns = ['category', 'order_count']

print("Orders by Category:")
print(orders_by_category.to_string(index=False))

top_cat = orders_by_category.iloc[0]
print(f"\n✅ Answer: '{top_cat['category']}' with {top_cat['order_count']} orders")

### Q3 — Who are the top 5 customers by total spending?

In [ ]:
top_customers = (
    delivered
    .groupby(['customer_id', 'customer_name'])['net_revenue']
    .sum()
    .reset_index()
    .nlargest(5, 'net_revenue')
    .reset_index(drop=True)
)
top_customers.columns = ['customer_id', 'customer_name', 'total_spent']
top_customers.index = top_customers.index + 1  # rank starts at 1

print("Top 5 Customers by Spending:")
print(top_customers[['customer_name', 'total_spent']].to_string())

### Q4 — Which month had the highest and lowest revenue?

In [ ]:
monthly_revenue = (
    delivered
    .groupby(['month', 'month_name'])['net_revenue']
    .sum()
    .reset_index()
    .sort_values('month')
)
monthly_revenue.columns = ['month_num', 'month', 'revenue']

print("Monthly Revenue:")
print(monthly_revenue[['month', 'revenue']].to_string(index=False))

best_month  = monthly_revenue.loc[monthly_revenue['revenue'].idxmax()]
worst_month = monthly_revenue.loc[monthly_revenue['revenue'].idxmin()]

print(f"\n✅ Highest: {best_month['month']} — ₹{best_month['revenue']:,.0f}")
print(f"✅ Lowest:  {worst_month['month']} — ₹{worst_month['revenue']:,.0f}")

### Q5 — What is the average order value per region?

In [ ]:
avg_order_value = (
    delivered
    .groupby('region')['net_revenue']
    .agg(
        total_orders='count',
        total_revenue='sum',
        avg_order_value='mean',
    )
    .round(0)
    .sort_values('avg_order_value', ascending=False)
    .reset_index()
)

print("Average Order Value by Region:")
print(avg_order_value.to_string(index=False))

### Q6 — How many orders were returned, and what % of revenue do returns represent?

In [ ]:
total_orders   = len(master)
returned_orders = master['is_return'].sum()
return_rate    = returned_orders / total_orders * 100

total_revenue   = master[~master['is_return']]['net_revenue'].sum()
returned_revenue = master[master['is_return']]['net_revenue'].sum()
revenue_loss_pct = returned_revenue / (total_revenue + returned_revenue) * 100

print(f"Total orders:    {total_orders}")
print(f"Returned orders: {returned_orders} ({return_rate:.1f}% of all orders)")
print(f"\nRevenue from delivered orders: ₹{total_revenue:,.0f}")
print(f"Revenue lost to returns:       ₹{returned_revenue:,.0f} ({revenue_loss_pct:.1f}% of gross revenue)")

print("\nReturned orders detail:")
print(master[master['is_return']][['order_id','customer_name','product_name','net_revenue']].to_string(index=False))

### Q7 — Which day of the week sees the most orders?

In [ ]:
# Order days from Monday(0) to Sunday(6)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

orders_by_day = (
    master
    .groupby('day_of_week')['order_id']
    .count()
    .reindex(day_order)  # ensure weekday order
    .reset_index()
)
orders_by_day.columns = ['day_of_week', 'order_count']

print("Orders by Day of Week:")
print(orders_by_day.to_string(index=False))

busiest = orders_by_day.loc[orders_by_day['order_count'].idxmax()]
print(f"\n✅ Busiest day: {busiest['day_of_week']} with {busiest['order_count']} orders")

### Q8 — Quarterly Revenue Report (Pivot Table)

In [ ]:
# Pivot: rows = Region, columns = Quarter, values = net_revenue
quarterly_report = delivered.pivot_table(
    values='net_revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
    fill_value=0,
    margins=True,           # add row/column totals
    margins_name='TOTAL'
)

# Format numbers with comma separators
print("=== Quarterly Revenue Report (₹) ===")
print(quarterly_report.applymap(lambda x: f'{x:,.0f}'))

---

## Step 7: Final Summary

Present your findings in a clean summary — like you would in a real report.

In [ ]:
print("="*55)
print("        TECHMART 2024 SALES ANALYSIS — SUMMARY")
print("="*55)

total_net_rev = delivered['net_revenue'].sum()
print(f"\n📦 Total Delivered Orders  : {len(delivered)}")
print(f"💰 Total Net Revenue       : ₹{total_net_rev:,.0f}")
print(f"↩️  Return Rate             : {return_rate:.1f}%")

print(f"\n🏆 Top Region              : {revenue_by_region.iloc[0]['region']}")
print(f"🏆 Top Category            : {top_cat['category']} ({top_cat['order_count']} orders)")
print(f"🏆 Top Customer            : {top_customers.iloc[0]['customer_name']}")
print(f"📅 Best Month              : {best_month['month']} (₹{best_month['revenue']:,.0f})")
print(f"📅 Slowest Month           : {worst_month['month']} (₹{worst_month['revenue']:,.0f})")
print(f"📅 Busiest Day             : {busiest['day_of_week']}")

print("\n" + "="*55)

---

## What You Used in This Project

| Notebook | Concept Used | Where |
|---|---|---|
| NB 2 | DataFrame creation, `.head()`, `.info()` | Step 1, 2 |
| NB 3 | `drop_duplicates`, `pd.to_numeric`, `fillna`, `replace` | Step 3 |
| NB 4 | `.str.strip()`, `.str.title()`, `.dt.month`, `.dt.day_name()` | Step 3, 5 |
| NB 5 | `sort_values`, `nlargest` | Q1, Q3 |
| NB 6 | `pd.cut`, `np.where` | Step 5 |
| NB 7 | `groupby`, `agg`, named aggregations | Q1–Q5, Q7 |
| NB 8 | `pd.merge` (left join, chained) | Step 4 |
| NB 10 | `pivot_table` with `margins=True` | Q8 |

---

## Key Takeaways from This Project

1. **Real data is always messy.** Every column in this project had at least one problem.
2. **Cleaning comes before analysis.** We spent 3 full steps cleaning before asking a single business question.
3. **Merge order matters.** We merged orders → products → customers in a logical dependency order.
4. **Derived columns unlock analysis.** `net_revenue`, `quarter`, `is_return` — none existed in raw data.
5. **GroupBy + Pivot = most of your reports.** Almost every business question used one of these.

---

**This is exactly what a data analyst does every day. You are ready.**